In [13]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# AISDI - ćwiczenie 2
## [1] Wprowadzenie
Celem drugiego zadania z przedmiotu Algorytmy i Struktury Danych (AISDI) była implementacja oraz analiza wydajności algorytmów grafowych na przykładzie miejskiej sieci drogowej. Głównym zadaniem było zaimportowanie danych zawierających odległości oraz czasy przejazdu między węzłami w różnych porach dnia, a następnie wyznaczenie optymalnych tras na podstawie kryterium dystansu oraz czasu.

### [1.1] Zaimportowane biblioteki
Zgodnie z wymaganiami projektowymi, do reprezentacji grafu nie wykorzystano gotowych bibliotek (takich jak networkx). Graf został zaimplementowany od podstaw przy użyciu zmodyfikowanej listy sąsiedztwa.

Główną strukturą przechowującą dane jest słownik, którego kluczem jest identyfikator węzła (miasta), a wartością lista obiektów klasy Destination.
Zastosowane narzędzia i moduły:

- csv – do wczytywania i parsowania plików z danymi.

- destination.py – autorska klasa reprezentująca krawędź grafu (cel podróży) wraz z jej wagami (dystans, czas w zależności od pory dnia).

- graph_logic.py – moduł zawierający implementację algorytmów trasowania.

- time – do przeprowadzania pomiarów wydajności i czasu kompilacji algorytmów.

In [26]:
import csv
from pprint import pprint
from destination import Destination
from graph_logic import *
import time

### [1.2] Analiza danych wejściowych
Dane wejściowe pochodzą z pliku city_small.csv. Każdy wiersz reprezentuje krawędź grafu nieskierowanego (dwukierunkową drogę) wraz z zestawem wag. Struktura danych prezentuje się następująco:

In [27]:
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    for index, row in enumerate(small_cities):
        if index > 5:
            break
        print(row)

['od', 'do', 'odleglosc', 'czas_rano', 'czas_poludnie', 'czas_wieczor']
['0', '9', '17.1', '19', '11', '14']
['0', '20', '6.84', '11', '5', '8']
['1', '11', '9.4', '22', '12', '12']
['1', '21', '8.12', '13', '11', '15']
['1', '22', '19.55', '33', '20', '20']


Konstrukcja grafu odbywa się poprzez iterację po pliku CSV i dwukrotne dodanie krawędzi (od A do B oraz od B do A) do słownika pathways za pomocą metody create_destination().


## [2] Wyznaczanie najkrótszej ścieżki
W pierwszej fazie badań przetestowano działanie klasycznego algorytmu Dijkstry, szukającego najkrótszej ścieżki od wybranego wierzchołka do pozostałych węzłów na podstawie wag odległościowych. Zbadano zachowanie algorytmu w zależności od rozmiaru grafu.

### [2.1] Małe miasto
Algorytm dla małej siatki skrzyżowań odnalazł trasy błyskawicznie. W wynikach prawidłowo zidentyfikowano braki połączeń między odizolowanymi węzłami (oznaczone wartością inf), co świadczy o tym, że badany podgraf nie jest w pełni spójny.

In [28]:

small_pathways = import_city("data/city_small.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(49, small_pathways)
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra is {total_time:.04f}")


Distance from 11 to 20 is inf
Distance from 29 to 26 is inf
Distance from 46 to 48 is 23.55
Distance from 9 to 16 is inf
Distance from 34 to 21 is 9.02
Distance from 25 to 46 is 49.14
Distance from 29 to 20 is inf
Distance from 11 to 43 is 77.26
Distance from 31 to 43 is 48.98
Distance from 16 to 43 is inf

Total time for small city dijkstra is 0.0004


### [2.2] Średnie miasto
Dla średniego zbioru danych zauważono oczekiwany wzrost czasu wykonywania algorytmu, związany z większą liczbą krawędzi poddawanych procesowi relaksacji.

In [29]:
mid_pathways = import_city("data/city_mid.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(199, mid_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for mid city dijkstra is {total_time:.04f}")

Distance from 42 to 170 is 41.10
Distance from 156 to 23 is 61.46
Distance from 165 to 59 is 61.23
Distance from 101 to 116 is 60.76
Distance from 145 to 177 is 83.42
Distance from 28 to 90 is 12.82
Distance from 167 to 189 is 96.48
Distance from 7 to 194 is 41.75
Distance from 27 to 136 is 38.48
Distance from 65 to 165 is 4.74

Total time for mid city dijkstra is 0.0027


### [2.3] Duże miasto 

In [40]:
big_pathways = import_city("data/city_big.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(999, big_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for big city dijkstra is {total_time:.04f}")

Distance from 794 to 5 is 33.24
Distance from 208 to 620 is 49.25
Distance from 677 to 688 is 30.19
Distance from 878 to 702 is 61.58
Distance from 188 to 370 is 12.96
Distance from 630 to 993 is 15.10
Distance from 654 to 483 is 78.50
Distance from 824 to 653 is 16.47
Distance from 622 to 828 is 52.43
Distance from 928 to 345 is 34.56

Total time for big city dijkstra is 0.0299


Na tym etapie porównamy średni czas podróży 
Algorytm wykonuje się stosunkowo szybko.
### Czas
- zaprezentowano algortym osobnej dijstry dla czasu
- liczy od w trakcie godzine/czas obecny dla grafu
- w trakcie iteracji po grafie zmienia godzine i od niej pobiera odpowiedni czas

In [30]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(49, small_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 20 to 4 is inf
Arrival time from 25 to 42 is 07:36
Arrival time from 18 to 2 is 07:53
Arrival time from 20 to 42 is inf
Arrival time from 45 to 27 is 10:34
Arrival time from 19 to 47 is 10:09
Arrival time from 22 to 0 is inf
Arrival time from 39 to 34 is 07:42
Arrival time from 27 to 42 is 08:59
Arrival time from 35 to 6 is inf

Total time for small city dijkstra time is 0.0005
This simulation is for starting time equal 07:30


### Komiwojazer:
Ten etap był najtrudniejszy do zrealizowania, po pierwsze w wielu przypadkach dotyczących grafu nie mam możliwości przejscia z jednegoe miasta do drugiego bezposrednio, dlatego trzeba zawracać. Dodatkowo niektóre miasta  tworzą graf, który jest odizolowany od głownego grafu, co powoduje, że klaysyczne rozwiązanie problemu kowimojażera staje się niemożliwe, a do tego należy wspomnieć, że tym przypadku została użyta heurystyka, bo dla zwykłego komputeraz obliczenie poprawnej najkrótszej drogi zajmuje n! czasu, dla kilku miast komputer szybko znajdzie najlepszą trasę, ale dla kilkudziesięciu miast potrzebny czas wynosi więcej niż czas istnienie wszechświata. W tym przypadku została użyta najprostrza heurystyka, która nie jest optymalna.

Tak prezentują się wyniki heurystyki, w której to algortym szuka najkrótszej drogi dla grafu małego miasta:

In [31]:
start_time = time.perf_counter()
start_city = "21"
road, distance = distance_of_travelling_salesman(small_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 21 are: 
['21', '39', '1', '34', '25', '42', '11', '4', '33', '44', '13', '6', '22', '23', '47', '29', '5', '8', '45', '28', '17', '40', '24', '37', '46', '38', '43', '48', '30', '19', '14', '18', '10', '15', '32', '2', '27', '7', '31', '49', '12', '36', '21']
and the distance is 631.46
It took programme 0.028657541959546506 for solving travelling salesman problem


## Średnie Miasto